# DevSecOps Automation with Jira and GitHub Integration

This notebook demonstrates the setup and operation of automated DevSecOps tasks between Jira and GitHub using the Cerebrum agentic framework.

## 1. Environment Setup

In [ ]:
import sys
import os
base_path = '/path/to/cerebrum'
sys.path.append(base_path)

import requests
import json
from src.server.controllers.jira_controller import JiraController
from src.server.models.jira_config import JiraConfig
from src.server.tools.jira_service import JiraService
from src.server.controllers.config_loader import load_config
from src.server.models.tasks_manager import TasksManager
from src.server.models.task import Task
from src.server.models.message import Message

## 2. Load Configuration and Create TasksManager

In [ ]:
# Base URL for the Cerebrum API
base_url = "http://localhost:8080"

# Health check
response = requests.get(f"{base_url}/health_check")
assert response.status_code == 200, "Cerebrum API is not responding"
print("Cerebrum API is running and healthy.")

In [ ]:
# Create TasksManager for DevSecOps automation
tasksmanager_data = {
    "name": "DevSecOps Automation Manager",
    "model": "gpt-4",
    "description": "TasksManager for DevSecOps automation tasks",
}

tasksmanager_response = requests.post(f"{base_url}/tasksmanager", json=tasksmanager_data)
assert tasksmanager_response.status_code == 201, "Failed to create TasksManager"
tasksmanager_id = tasksmanager_response.json()["id"]
print(f"TasksManager created with ID: {tasksmanager_id}")

In [ ]:
# Load DevSecOps configuration
config_id = await load_config('config/devsecops_config.json')
print(f"Configuration loaded with ID: {config_id}")

# Create task using the configuration
task_data = {
    "name": "DevSecOps Automation",
    "description": "Automate DevSecOps tasks for microservices communication",
    "tasksmanager_id": tasksmanager_id,
    "crewai_info_id": config_id,
    "messages": [
        Message(
            role="system",
            content="You are a DevSecOps specialist responsible for implementing secure service-to-service communication in a microservices architecture."
        ).dict()
    ]
}

task_response = requests.post(f"{base_url}/task", json=task_data)
assert task_response.status_code == 201, "Failed to create task"
task_id = task_response.json()["id"]
print(f"Task created with ID: {task_id}")

## 3. Setup Jira Integration

In [ ]:
# Create Jira configuration
jira_config = JiraConfig(
    project_key="DSO",
    jira_url=os.getenv('JIRA_URL'),
    jira_username=os.getenv('JIRA_USERNAME'),
    jira_api_token=os.getenv('JIRA_API_TOKEN'),
    task_id=task_id,  # Use the task_id we just created
    schedule_interval=5  # Check every 5 minutes
)

# Create JiraService
jira_service = JiraService(
    jira_config.jira_url,
    jira_config.jira_username,
    jira_config.jira_api_token
)

# Create JiraController
jira_controller = JiraController(jira_config, jira_service)

## 4. Create and Process Test Ticket

In [ ]:
# Create a test ticket
test_ticket = {
    'fields': {
        'project': {'key': 'DSO'},
        'summary': 'Add ingress call from payment-service to access user-service',
        'description': 'Need to access user service from payment service for handling user information during payments.',
        'issuetype': {'name': 'Task'},
        'labels': ['DevSecOps']
    }
}

created_ticket = await jira_service.create_issue(test_ticket)
print(f"Created test ticket: {created_ticket['key']}")

In [ ]:
# Process the ticket
await jira_controller.process_devsecops_ticket(created_ticket)
print(f"Ticket processed: {created_ticket['key']}")

## 5. Monitor Changes

In [ ]:
# Get ticket status and comments
ticket_details = await jira_service.get_issue(created_ticket['key'])
print(f"Ticket Status: {ticket_details['fields']['status']['name']}")
print("\nComments:")
for comment in ticket_details['fields']['comment']['comments']:
    print(f"- {comment['body']}")

# Get related GitHub PR if available
if 'Pull Request created:' in ticket_details['fields']['description']:
    pr_url = ticket_details['fields']['description'].split('Pull Request created: ')[1].split('\n')[0]
    print(f"\nGitHub Pull Request: {pr_url}")

## 6. Cleanup

In [ ]:
# Optional: Clean up resources if needed
# Note: In production, you might want to keep these resources

# Delete the task
response = requests.delete(f"{base_url}/task/{task_id}")
print(f"Task cleanup status: {response.status_code}")

# Delete the tasksmanager
response = requests.delete(f"{base_url}/tasksmanager/{tasksmanager_id}")
print(f"TasksManager cleanup status: {response.status_code}")